# Merge smoke test

Verifies the merged files + SL trained weights load and run end to end.
Run top to bottom; each cell prints ✓ on success. **Launch Jupyter from your repo root**
(`~/semester_6/chess_bot`) and set `WEIGHT_PATH` below.

## 0. Config

In [1]:
import os, sys

REPO_ROOT   = os.path.abspath(".")        # must be your repo root
WEIGHT_PATH = "reinforcement_learning/networks/weights/sl_best.weights.h5"  # <-- set this
LOOKUP_PATH = "reinforcement_learning/move_lookup.json"

# These MUST match the trained file
NUM_RES_BLOCKS = 20
NUM_FILTERS    = 256
SE_RATIO       = 8

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

assert os.path.exists(LOOKUP_PATH), f"move_lookup.json not found at {LOOKUP_PATH}"
assert os.path.exists(WEIGHT_PATH), f"weight file not found at {WEIGHT_PATH}"
print("✓ repo root + files present")

✓ repo root + files present


## 1. Imports (catches import-prefix / missing-file issues)

In [2]:
import chess
import numpy as np

from reinforcement_learning.helpers.converter import Converter
from reinforcement_learning.helpers.move_encoding import gather_indices_from_lookup_path
from reinforcement_learning.monte_carlo_tree_search.nodes_and_edges_v2 import Node, Edge
from reinforcement_learning.monte_carlo_tree_search.mcts_v2 import MCTS, SelfPlayGame
from reinforcement_learning.networks.big_network import BigNetwork
print("✓ all modules import")

I0000 00:00:1782411209.123740   21209 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782411209.130256   21209 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1782411209.828635   21209 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782411212.357320   21209 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

✓ all modules import


## 2. Converter + 20-plane input tensor

In [3]:
converter = Converter(lookup_path=LOOKUP_PATH)
assert len(converter.lookup) == 1858, f"lookup has {len(converter.lookup)} moves, expected 1858"

board = chess.Board()
t = converter.board_to_input_tensor(board)
assert t.shape == (8, 8, 20), t.shape
assert t[:, :, 19].all(), "edge plane (19) should be all ones"
assert t[1, :, 0].sum() == 8, "white pawns (friendly, plane 0, rank index 1) should be 8"
assert t[0, 0, 16] == 0, "side-to-move plane should be 0 for white to move"
print("✓ converter + input tensor look correct", t.shape, t.dtype)

✓ converter + input tensor look correct (8, 8, 20) float16


## 3. Build network + load his weights  ← the key check

In [4]:
net = BigNetwork(
    num_res_blocks=NUM_RES_BLOCKS,
    num_filters=NUM_FILTERS,
    se_ratio=SE_RATIO,
    lookup_path=LOOKUP_PATH,
)
print("✓ network built — gather encoding validated against move_lookup")

try:
    net.load(WEIGHT_PATH)
    print("✓ weights loaded — hyperparameters + lookup align with the trained file")
except Exception as e:
    print("✗ WEIGHT LOAD FAILED — mismatch in architecture or lookup.")
    print("   Check NUM_RES_BLOCKS/NUM_FILTERS/SE_RATIO and that move_lookup.json")
    print("   is byte-identical to your colleague's.")
    raise

E0000 00:00:1782411243.399647   21209 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


✓ network built — gather encoding validated against move_lookup
✓ weights loaded — hyperparameters + lookup align with the trained file


/home/timle/semester_6/chess_bot/.venv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 512 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


## 4. Single + batched inference

In [5]:
policy, value = net.predict(t)
assert policy.shape == (1858,), policy.shape
assert value.shape == (3,), value.shape
assert np.isfinite(policy).all(), "policy logits must be finite"
assert abs(value.sum() - 1.0) < 1e-3, f"WDL should sum to 1, got {value.sum()}"
print(f"✓ predict() ok | WDL={value.round(3)} (W/D/L)")

# Masked legal distribution should sum to 1 with exactly len(legal) nonzeros
dist = converter.mask_illegal_moves(board, policy).numpy()
assert abs(dist.sum() - 1.0) < 1e-3, dist.sum()
assert int((dist > 0).sum()) == board.legal_moves.count()
print(f"✓ legal-move softmax ok | {int((dist>0).sum())} legal moves")

# Batched inference on a few positions
b2 = board.copy(); b2.push_san("e4")
b3 = b2.copy();   b3.push_san("c5")
pol_b, val_b = net.predict_batch([converter.board_to_input_tensor(x) for x in (board, b2, b3)])
assert pol_b.shape == (3, 1858) and val_b.shape == (3, 3)
print("✓ predict_batch() ok", pol_b.shape, val_b.shape)

✓ predict() ok | WDL=[0.288 0.417 0.296] (W/D/L)
✓ legal-move softmax ok | 20 legal moves
✓ predict_batch() ok (3, 1858) (3, 3)


## 5. MCTS returns a legal move (batched search)

In [6]:
import time
t0 = time.perf_counter()
move = net.search_for_best_move(board, num_simulations=30, batch_size=8)
assert move in board.legal_moves, f"{move} is not legal!"
print(f"✓ search_for_best_move -> {move.uci()} ({time.perf_counter()-t0:.1f}s for 30 sims)")

✓ search_for_best_move -> e2e4 (0.4s for 30 sims)


## 6. One micro self-play game (full pipeline)

In [7]:
mcts = MCTS(network=net, converter=converter, num_simulations=8)
game = SelfPlayGame(
    mcts, temperature_threshold=3, max_moves=6,
    resign_threshold=None, search_batch_size=4,
)
samples = game.play()

assert len(samples) > 0, "no training samples produced"
s = samples[0]
assert s["board_tensor"].shape == (8, 8, 20)
assert s["policy_target"].shape == (1858,)
assert abs(s["policy_target"].sum() - 1.0) < 1e-3
assert s["value_target"] in (-1.0, 0.0, 1.0)
assert game.record and "pgn" in game.record and game.record["pgn"]
print(f"✓ self-play ok | {len(samples)} samples | result {game.record['result']}")
print("--- PGN ---")
print(game.record["pgn"])

  Game over after 6 moves: 1/2-1/2
✓ self-play ok | 6 samples | result 1/2-1/2
--- PGN ---
[Event "self-play"]
[Site "?"]
[Date "????.??.??"]
[Round "?"]
[White "?"]
[Black "?"]
[Result "1/2-1/2"]

1. c4 e6 2. d4 Nf6 3. Nf3 d5 1/2-1/2


## ✓ If every cell passed

The merge is sound: files import, the 20-plane encoding works, his weights load (so
architecture + `move_lookup.json` + gather encoding are all aligned), inference and
batched MCTS run, and the full self-play path produces samples + a PGN.